# NVML：用 Python 读 GPU 状态

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-06-nvml.ipynb)

直接对接 HAMi 源码。用 pynvml 调用 NVML 接口，拿到温度、功耗、显存、利用率、UUID、MIG 能力，这些正是 HAMi 设备发现与配额管理依赖的同一套 API。

本 notebook 配合教程章节 [NVML：用 Python 读 GPU 状态](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

前面五个 Lab 你一直用 `nvidia-smi` 命令行看 GPU。但在真实平台里，HAMi、DCGM、Prometheus、Kubernetes device plugin 这些组件都需要**程序化地**读 GPU 状态，而它们用的都是同一套底层：**NVML**。本 Lab 让你亲手调 NVML 的 Python 绑定 `pynvml`，拿到温度、功耗、显存、利用率、UUID，再把这些字段和 `nvidia-smi` 的输出、和 HAMi 源码里的调用一一对应。做完这一章，你再看 HAMi 的 device-plugin 代码就不会陌生。

![nvidia-smi、pynvml、DCGM Exporter、Device Plugin 和 HAMi 都通过 NVML（libnvidia-ml.so）经内核驱动读取 GPU 状态](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/nvml-consumers.svg)

## 第 1 步：安装并初始化

In [ ]:
!pip install -q nvidia-ml-py
from pynvml import *
nvmlInit()
print("NVML version  :", nvmlSystemGetNVMLVersion())
print("driver version:", nvmlSystemGetDriverVersion())

实测输出：

```text
NVML version  : 13.580.82.07
driver version: 580.82.07
```

NVML 随驱动一起安装，所以它的版本号就是“CUDA 大版本 13 + 驱动版本 580.82.07”。`nvmlInit()` 不报错，说明 Python 已经能通过 NVML 和驱动对话。

> **装 nvidia-ml-py，不要装 pynvml**
>
> PyPI 上的 `pynvml` 包已被 NVIDIA 标记为废弃，官方维护的是 **`nvidia-ml-py`**，两者都提供 `import pynvml` 这个模块名。新版的字符串返回值已经是 `str`，不再是 `bytes`。HAMi 的 Go 代码用的是同一套 NVML C API 的 Go 绑定 `go-nvml`。

## 第 2 步：拿到设备句柄和身份

In [ ]:
handle = nvmlDeviceGetHandleByIndex(0)
print("name :", nvmlDeviceGetName(handle))
print("uuid :", nvmlDeviceGetUUID(handle))
print("index:", nvmlDeviceGetIndex(handle))

实测输出（UUID 每张卡不同）：

```text
name : Tesla T4
uuid : GPU-2174719e-76ca-d370-94ea-2132503039b7
index: 0
```

`index` 是这张卡在本机的编号，换台机器、重启后可能变；`uuid` 是烧在卡上的全球唯一身份，永远不变。平台组件识别 GPU 时都用 UUID，而不是 index。

> **为什么 UUID 这么重要**
>
> HAMi 在调度时识别 GPU 的核心字段就是 **UUID**。当多个 Pod 共享一张卡时，HAMi 在 Pod 注解里写的就是 `{UUID},{vendor},{VRAM},{compute%}`（见 [L1 HAMi 第一份共享作业](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/single-gpu/hami-first-share/)）。你现在拿到的这个 `GPU-...`，就是那张被切分/共享的卡的唯一身份。

## 第 3 步：显存信息

In [ ]:
mem = nvmlDeviceGetMemoryInfo(handle)
print("total MiB :", round(mem.total/1024**2))
print("used  MiB :", round(mem.used/1024**2))
print("free  MiB :", round(mem.free/1024**2))

实测输出：

```text
total MiB : 15360
used  MiB : 450
free  MiB : 14910
```

`total` 是 15360 MiB（15 GiB），`used + free = total`。新开的 runtime 里 `used` 应该接近 0；实测时前一个 Lab 的进程还没完全退出，所以多了 450 MiB。这个数和 `nvidia-smi --query-gpu=memory.used` 的口径**完全一致**：[Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 里你看到的“caching allocator 攥着显存不还”现象，用 NVML 读出来同样存在。HAMi 的 `nvidia.com/gpumem` 配额正是基于这层口径。

## 第 4 步：利用率与功耗、温度

In [ ]:
util = nvmlDeviceGetUtilizationRates(handle)
print("gpu util % :", util.gpu)
print("mem util % :", util.memory)   # 注意：这是显存控制器利用率，不是占用率

print("power W    :", nvmlDeviceGetPowerUsage(handle)/1000.0, "/", nvmlDeviceGetPowerManagementLimit(handle)/1000.0)
print("temp C     :", nvmlDeviceGetTemperature(handle, NVML_TEMPERATURE_GPU))
print("perf state :", nvmlDeviceGetPerformanceState(handle))   # P8 空载 / P0 满载
try:
    print("fan %      :", nvmlDeviceGetFanSpeed(handle))
except NVMLError_NotSupported:
    print("fan %      : not supported (T4 是被动散热，没有风扇)")

实测输出：

```text
gpu util % : 0
mem util % : 0
power W    : 9.168 / 70.0
temp C     : 37
perf state : 8
fan %      : not supported (T4 是被动散热，没有风扇)
```

逐行解读：GPU 空闲，所以两个利用率都是 0；功耗 9 W，上限 70 W（T4 是低功耗推理卡）；温度 37 °C；性能状态 P8 是最省电的空载档，跑满时会升到 P0；T4 靠服务器机箱风道散热，没有自己的风扇，所以读不到风扇转速。

`NVMLError_NotSupported` 是你读 HAMi / DCGM 代码时会反复遇到的异常：同一套 API 要覆盖从消费卡到数据中心卡的所有产品，很多字段在某些卡上就是不存在的。

注意区分两个“显存”指标：

- `memory.used` / `total`（第 3 步）：**显存占用**（字节）；
- `utilization.memory`（第 4 步）：**显存控制器忙闲比**（百分比）。

后者是 [Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) “纯拷贝时 GPU-Util 低、但显存其实很忙”的那个“忙”，它和 GPU-Util 是两个独立维度。

## 第 5 步：谁在用这张卡

In [ ]:
import torch; torch.zeros(1, device="cuda")   # 让当前进程持有一个 CUDA context
procs = nvmlDeviceGetComputeRunningProcesses(handle)
print("compute processes:", len(procs))
for p in procs:
    print("  pid", p.pid, "used MiB", None if p.usedGpuMemory is None else round(p.usedGpuMemory/1024**2))

实测输出（PID 每次不同）：

```text
compute processes: 1
  pid 11987 used MiB 120
```

列表里唯一的进程就是当前 notebook 的 Python 内核，它占的 120 MiB 正是 [Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 里看到的 CUDA context。在 Colab 上这个 PID 和 `os.getpid()` 一致（2026-11-08 验证过）。

这是 HAMi 和 DCGM 用来回答“哪个容器在用哪张卡、用了多少”的接口。但在 Kubernetes 的普通容器里，你常常会看到 PID 对不上、甚至列表为空：NVML 返回的是**宿主机**上的进程号，而容器有自己的一套进程号。这个“容器里看不清自己的 GPU 进程”问题，正是 Kubernetes 上 GPU 监控要靠 DCGM Exporter 做 Pod 归属映射的原因。

## 第 6 步：MIG 能力探测（为 [Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 铺垫）

In [ ]:
try:
    mode, pending = nvmlDeviceGetMigMode(handle)
    print("MIG mode:", mode, "pending:", pending)
except NVMLError_NotSupported:
    print("MIG not supported on this GPU")

实测输出：

```text
MIG not supported on this GPU
```

T4 不支持 MIG（需要 Ampere 及以上的 A100 / A30 / H100 等）。这本身是个有用的事实：**不是所有卡都能做硬隔离 MIG**，这正是 [Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 里“为什么要有 MPS / time-slicing”的起点。

## 验证

完成后你应该能：

- 用 `pynvml` 拿到 name / uuid / memory / util / power / temp；
- 区分“显存占用”和“显存控制器利用率”；
- 解释 UUID 在 HAMi 共享调度里的角色；
- 知道 T4 为何查不到 MIG mode。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| `nvmlInit()` 报 `Driver not loaded` | runtime 是 CPU | 切到 T4 |
| `nvmlDeviceGetFanSpeed` 抛 `NVMLError_NotSupported` | T4 无风扇 | try/except 包住（代码已处理） |
| 字符串字段返回 `b'...'` | 装到了老版 `pynvml` | `pip install -U nvidia-ml-py` |
| `GetComputeRunningProcesses` 返回空或 PID 对不上 | 容器 PID 命名空间隔离（Colab 上能看到，K8s 普通容器里常见） | 正常，记录为“容器内不可见” |
| MIG 调用报 `Not Supported` | T4 架构不支持 | 正常，记录为“不支持” |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。